## Feature Engineering
The process of creating, transforming, or selecting input variables (features) from raw data to improve the performance of machine learning models. It often involves techniques like scaling, encoding categorical data, creating new features from existing ones, or reducing dimensionality. Essentially, it’s about turning raw data into meaningful inputs that help models learn patterns more effectively.

Create meaningful features:
- One-hot encoding
- Extract date features (weekday, month, hour…)
- Domain logic features
- Combine or derive features (ratios, interactions)
- Transform skew (log transform)
- Text preprocessing (tokenisation, embeddings)

Here we will be using pandas to handle storing and processing the data

In [26]:
import pandas as pd

df = pd.read_csv("data/nba_games.csv",index_col=0)

In [27]:
df.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,usg%_max_opp,ortg_max_opp,drtg_max_opp,team_opp,total_opp,home_opp,season,date,won,target
2630,240.0,39.0,75.0,0.520,13.0,19.0,0.684,22.0,22.0,1.000,...,29.7,150.0,133.0,UTA,104,0,2017,2016-10-25,True,0
2631,240.0,40.0,82.0,0.488,8.0,24.0,0.333,16.0,16.0,1.000,...,39.2,220.0,120.0,POR,113,1,2017,2016-10-25,False,1
2632,240.0,40.0,85.0,0.471,7.0,33.0,0.212,13.0,18.0,0.722,...,36.2,263.0,112.0,SAS,129,0,2017,2016-10-25,False,1
2633,240.0,47.0,98.0,0.480,12.0,24.0,0.500,23.0,26.0,0.885,...,31.8,139.0,143.0,GSW,100,1,2017,2016-10-25,True,1
2634,240.0,45.0,94.0,0.479,13.0,35.0,0.371,14.0,19.0,0.737,...,36.8,153.0,125.0,NYK,88,0,2017,2016-10-25,True,1


In [28]:
def remove_cols(df, cols):
    for col in cols:
        del df[col]
    return df

In [29]:
removed_columns = ["season", "date", "won", "target", "team", "team_opp"]
selected_columns = df.columns[~df.columns.isin(removed_columns)]

In [23]:
def find_team_averages(team):
    #rolling = team.rolling(10).mean()
    rolling = team.shift().rolling(10, min_periods=1).mean()
    return rolling

df_rolling = df[list(selected_columns) + ["won", "team", "season"]]
df_rolling = df_rolling.groupby(["team", "season"], group_keys=False).apply(find_team_averages)

In [36]:
def find_team_averages(group):
    # Compute rolling mean with a 10-game window, shifted so current row isn’t included
    return group.shift().rolling(10, min_periods=1).mean()

# Select relevant columns
cols_to_roll = list(selected_columns) + ["won"]

# Apply rolling per team per season
df_rolling = df.groupby(["team", "season"], group_keys=False)[cols_to_roll].apply(find_team_averages)

In [37]:
df_rolling.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,ast%_max_opp,stl%_max_opp,blk%_max_opp,tov%_max_opp,usg%_max_opp,ortg_max_opp,drtg_max_opp,total_opp,home_opp,won
2630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2631,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2634,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
rolling_cols = [f"{col}_10" for col in df_rolling.columns]
df_rolling.columns = rolling_cols

In [7]:
df = pd.concat([df, df_rolling], axis=1)

In [8]:
df = df.dropna() #Drop all null/na values

In [9]:
def shift_col(team, col_name):
    next_col = team[col_name].shift(-1)
    return next_col

def add_col(df, col_name):
    return df.groupby("team", group_keys=False).apply(lambda x: shift_col(x, col_name))

df["home_next"] = add_col(df, "home")
df["team_opp_next"] = add_col(df, "team_opp")
df["date_next"] = add_col(df, "date")

df = df.copy()

In [10]:
full = df.merge(
    df[rolling_cols + ["team_opp_next", "date_next", "team"]], 
    left_on=["team", "date_next"], 
    right_on=["team_opp_next", "date_next"]
)

In [12]:
full["date"] = pd.to_datetime(full["date"])
full["date_next"] = pd.to_datetime(full["date_next"])

In [13]:
full["days_rested"] = (full["date_next"] - full["date"]).dt.days

In [14]:
full = full[full["days_rested"] <= 4]

In [15]:
removed_columns = list(full.columns[full.dtypes == "object"]) + removed_columns

In [16]:
selected_columns = full.columns[~full.columns.isin(removed_columns)]

In [18]:
full.head()

,mp,fg,fga,fg%,3p,3pa,3p%,ft,fta,ft%,...,usg%_max_opp_10_y,ortg_max_opp_10_y,drtg_max_opp_10_y,total_opp_10_y,home_opp_10_y,won_10_y,season_10_y,team_opp_next_y,team_y,days_rested
0,240.0,36.0,79.0,0.456,6.0,18.0,0.333,24.0,27.0,0.889,...,25.3,150.0,102.0,107.0,0.0,0.0,2017.0,SAS,NOP,2
1,240.0,38.0,76.0,0.500,8.0,23.0,0.348,15.0,24.0,0.625,...,28.1,130.0,116.0,96.0,1.0,1.0,2017.0,BOS,CHO,2
2,240.0,32.0,84.0,0.381,9.0,23.0,0.391,16.0,21.0,0.762,...,46.6,180.0,110.0,97.0,1.0,1.0,2017.0,LAL,OKC,2
3,240.0,35.0,90.0,0.389,4.0,16.0,0.250,17.0,18.0,0.944,...,45.2,127.0,112.0,102.0,1.0,1.0,2017.0,TOR,DEN,3
4,240.0,33.0,79.0,0.418,12.0,32.0,0.375,16.0,22.0,0.727,...,27.8,176.0,110.0,108.0,0.0,0.0,2017.0,CLE,ORL,1


In [19]:
full.to_pickle('df_nba_games.pkl')